# Text Analysis Using SVD, NMF, and Householder Transformation

This notebook analyzes English news text using TF-IDF representation, Singular Value Decomposition (SVD), Non-negative Matrix Factorization (NMF), and Householder transformation.

## 1. Import Libraries

Import the libraries required for text preprocessing, matrix decomposition, and numerical operations.

In [ ]:
import re
import numpy as np
import nltk

from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from scipy.sparse.linalg import svds
from scipy.sparse import issparse

nltk.download('punkt')
nltk.download('punkt_tab')

## 2. Text Preprocessing

The input text file contains news documents. Preprocessing consists of removing punctuation marks and numerical characters, tokenizing the text, and applying Porter stemming. English stop words are removed later during TF-IDF vectorization.

In [ ]:
class DataProcessor:
    def __init__(self, path):
        self.data = []
        self.cleaned_data = []
        self.prepared_data = []

        self.read_data(path)
        self.clean_text()
        self.stemming()

    def read_data(self, path):
        with open(path, 'r', encoding='utf-8') as file:
            self.data = file.readlines()

    def clean_text(self):
        for document in self.data:
            text = re.sub(r'[0-9\u0660-\u0669\u06F0-\u06F9]', '', document)
            text = re.sub(r'[^a-zA-Z\s]', ' ', text)
            text = re.sub(r'\s+', ' ', text).strip()
            self.cleaned_data.append(text)

    def stemming(self):
        stemmer = PorterStemmer()

        for document in self.cleaned_data:
            words = word_tokenize(document)
            stemmed_words = [stemmer.stem(word) for word in words]
            self.prepared_data.append(' '.join(stemmed_words))

        return self.prepared_data

    def get_data(self):
        return self.prepared_data

## 3. Load and Process the Dataset

Load the news documents from the input file and apply the preprocessing pipeline. Each line is treated as one document.

In [ ]:
path = '/content/isna-en.txt'

all_data = DataProcessor(path)
data_list = all_data.get_data()

print('Number of documents:', len(data_list))
print('\nExample of preprocessed text:')
print(data_list[0])

## 4. TF-IDF Matrix Construction

Convert the preprocessed documents into a TF-IDF matrix. Rows represent documents and columns represent terms. TF-IDF assigns higher weights to terms that are frequent in a document but relatively uncommon across the entire collection.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(data_list)

feature_names = vectorizer.get_feature_names_out()

print('TF-IDF matrix shape:', X.shape)
print('Number of terms:', len(feature_names))
print('\nFirst 20 terms:')
print(feature_names[:20])

## 5. Method 1: Singular Value Decomposition (SVD)

Apply SVD to the TF-IDF matrix. The matrix is decomposed into three components:

\[X = U\Sigma V^T\]

The matrices contain document-related and term-related latent representations. A truncated decomposition is used to obtain the leading components without computing the complete decomposition.

In [ ]:
n_components = min(3, min(X.shape) - 1)

if n_components < 1:
    raise ValueError('The TF-IDF matrix must have at least two rows and two columns.')

U, s, VT = svds(X, k=n_components)

# Arrange singular values in descending order
order = np.argsort(s)[::-1]
s = s[order]
U = U[:, order]
VT = VT[order, :]

print('U shape:', U.shape)
print('Singular values:', s)
print('VT shape:', VT.shape)

## 6. Method 2: Non-negative Matrix Factorization (NMF)

Apply NMF to the TF-IDF matrix. NMF approximates the original matrix as the product of two non-negative matrices:

\[X \approx CD\]

Here, C represents document-topic weights, while D represents topic-term weights. The number of latent components is set to 3.

In [ ]:
k = 3

model = NMF(
    n_components=k,
    init='random',
    random_state=0,
    max_iter=500
)

C = model.fit_transform(X)
D = model.components_

print('C shape:', C.shape)
print('D shape:', D.shape)
print('Approximation error:', model.reconstruction_err_)

## 7. Householder Transformation

A Householder transformation is an orthogonal reflection used to transform a vector into a direction aligned with the first coordinate axis. The transformation matrix is defined as:

\[H = I - 2\frac{uu^T}{u^Tu}\]

where u is constructed from the input vector. The resulting matrix is orthogonal and symmetric, meaning that \(H^T H = I\) and \(H^T = H\).

In [ ]:
def get_norm(x):
    return np.linalg.norm(x)


def householder_transformation(v):
    v = np.asarray(v, dtype=float).reshape(-1, 1)
    n = v.shape[0]

    norm_v = get_norm(v)

    if norm_v == 0:
        return np.eye(n)

    e1 = np.zeros((n, 1))
    e1[0, 0] = 1.0

    sign = 1.0 if v[0, 0] >= 0 else -1.0
    u = v + sign * norm_v * e1

    norm_u_squared = float(u.T @ u)

    if norm_u_squared == 0:
        return np.eye(n)

    H = np.eye(n) - 2.0 * (u @ u.T) / norm_u_squared

    return H

## 8. Select the Most Important Column of D

Calculate the Euclidean norm of each column of D. Select the column with the largest norm and exchange it with the first column. This places the selected component in the first position before applying the Householder transformation.

In [ ]:
column_norms = np.linalg.norm(D, axis=0)

I = np.argmax(column_norms)

print('Selected column index:', I)
print('Largest column norm:', column_norms[I])

# Exchange the selected column with the first column
D[:, [0, I]] = D[:, [I, 0]]

# Apply the same permutation to the corresponding columns of C
C[:, [0, I]] = C[:, [I, 0]]

print('Updated D shape:', D.shape)
print('Updated C shape:', C.shape)

## 9. Apply Householder Transformation to C and D

Construct the Householder matrix using the first column of D. Apply the orthogonal transformation to D and the corresponding inverse transformation to C. Since the Householder matrix is orthogonal, the matrix product CD is preserved.

In [ ]:
# Store the original NMF approximation
original_approximation = C @ D

# First column of D
c = D[:, 0].reshape(k, 1)

# Construct the Householder matrix
Q = householder_transformation(c)

# Update both factor matrices
D = Q.T @ D
C = C @ Q

print('Householder matrix shape:', Q.shape)
print('Updated D shape:', D.shape)
print('Updated C shape:', C.shape)

print('\nFirst column of updated D:')
print(D[:, 0])

## 10. Verify the Transformation

Check whether the Householder transformation preserves the NMF approximation. The difference between the original and updated products should be close to zero, apart from numerical precision errors.

In [ ]:
updated_approximation = C @ D

difference = np.linalg.norm(
    original_approximation - updated_approximation
)

print('Difference between the original and updated CD products:')
print(difference)

print('\nFirst column of C:')
print(C[:, 0])

## 11. Conclusion

In this notebook, news documents were preprocessed and represented using TF-IDF. SVD was applied to obtain a low-dimensional representation, while NMF decomposed the TF-IDF matrix into document-component and component-term matrices. Finally, a Householder transformation was applied to the NMF factor matrices while preserving their product.